# Stage 6 — FT Reranker
`Dense + BM25 → RRF → FT CE → Top-k → Base LLM`

Stage 5'ten tek fark: **FT Cross-Encoder**

## 0. GPU

In [1]:
import os, gc, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"
print('Input:', os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'YOK')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {torch.cuda.get_device_name(i)} | {p.total_memory/1024**3:.1f} GB')
else:
    print('GPU bulunamadi — Kaggle > Settings > Accelerator > T4 x2')

Input: ['datasets']
GPU 0: Tesla T4 | 14.6 GB
GPU 1: Tesla T4 | 14.6 GB


## 1. Kurulum

In [2]:
!pip install -q -U bitsandbytes sentence-transformers==3.4.1 faiss-cpu rank_bm25 rouge-score nltk pandas tqdm
import nltk; nltk.download('punkt', quiet=True)

# Versiyon kontrolü
import subprocess
result = subprocess.run(['pip', 'show', 'bitsandbytes'], capture_output=True, text=True)
print(result.stdout)
print('Kurulum tamam.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 89.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 111.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 109.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 110.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following d

## 2. Config

In [3]:
import os, json, re, math, gc, pickle
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict

@dataclass
class Config:
    SYSTEM_NAME  : str  = 'System6_FTReranker'
    SYSTEM_DESC  : str  = 'FT BGE-M3 + BM25 + RRF + FT CE + Base Qwen2.5-7B'
    DRIVE_DIR    : str  = '/kaggle/input/datasets/ardayildiz29/legalo'
    CORPUS_FILE  : str  = 'corpus.jsonl'
    GOLD_FILE    : str  = 'gold_benchmark.json'
    INDEX_DIR    : str  = '/kaggle/working/legal_rag/index_s6'
    MODEL_DIR    : str  = '/kaggle/input/datasets/ardayildiz29/legal-models-tr-finetuned'
    RESULTS_DIR  : str  = '/kaggle/working/legal_rag/results'
    EMBED_BASE   : str  = 'BAAI/bge-m3'
    EMBED_DEVICE : str  = 'cuda'
    EMBED_BATCH  : int  = 64
    TOP_K_DENSE     : int = 20
    TOP_K_SPARSE    : int = 20
    TOP_K_RERANK_IN : int = 20
    TOP_K_FINAL     : int = 5
    RRF_K           : int = 60
    RERANKER_BASE   : str  = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
    RERANKER_MAX_LEN: int  = 384
    LLM_MODEL       : str  = 'Qwen/Qwen2.5-7B-Instruct'
    MAX_NEW_TOKENS  : int  = 256
    BENCH_SIZE   : int = 240
    RANDOM_SEED  : int = 42

CFG = Config()
for d in [CFG.INDEX_DIR, CFG.MODEL_DIR, CFG.RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
with open(f'{CFG.RESULTS_DIR}/{CFG.SYSTEM_NAME}_config.json', 'w', encoding='utf-8') as f:
    json.dump(asdict(CFG), f, ensure_ascii=False, indent=2)
print(f'Sistem   : {CFG.SYSTEM_NAME}')
print(f'Aciklama : {CFG.SYSTEM_DESC}')
print('Config kaydedildi.')

Sistem   : System6_FTReranker
Aciklama : FT BGE-M3 + BM25 + RRF + FT CE + Base Qwen2.5-7B
Config kaydedildi.


## 3. Veri

In [4]:
import random
from tqdm import tqdm

DRIVE = Path(CFG.DRIVE_DIR)
for name, path in [('corpus', DRIVE/CFG.CORPUS_FILE), ('gold', DRIVE/CFG.GOLD_FILE)]:
    if not path.exists():
        raise FileNotFoundError(f'Eksik: {path}')
    print(f'  {name}: OK ({path.stat().st_size/1024**2:.1f} MB)')

def load_jsonl(p):
    rows = []
    with open(p, encoding='utf-8') as f:
        for ln, line in enumerate(f,1):
            line = line.strip()
            if line:
                try: rows.append(json.loads(line))
                except json.JSONDecodeError as e: raise ValueError(f'{p.name} satir={ln}: {e}')
    return rows

print('Corpus yukleniyor...')
CORPUS: List[Dict] = [{'id':r['id'],'text':r['text'].strip(),
                        'title':r.get('title',''),'metadata':r.get('metadata',{})}
                       for r in load_jsonl(DRIVE/CFG.CORPUS_FILE)]
print(f'Corpus: {len(CORPUS)} chunk')
corpus_id_set = {c['id'] for c in CORPUS}
assert len(corpus_id_set) == len(CORPUS), 'Duplicate corpus id!'

print('Benchmark yukleniyor...')
with open(DRIVE/CFG.GOLD_FILE, encoding='utf-8') as f:
    gold_raw = json.load(f)
BENCHMARK = []
for item in gold_raw:
    rel_ids = [s.get('source_id') or s.get('corpus_row_id')
               for s in item.get('gold_sources',[]) if s.get('source_id') or s.get('corpus_row_id')]
    q = item.get('question','').strip(); a = item.get('verified_answer','').strip()
    if q and a and rel_ids:
        BENCHMARK.append({'question_id':item.get('question_id'),'question':q,'answer':a,'relevant_ids':rel_ids})
random.seed(CFG.RANDOM_SEED); random.shuffle(BENCHMARK)
BENCHMARK = BENCHMARK[:CFG.BENCH_SIZE]
covered = sum(1 for b in BENCHMARK if any(r in corpus_id_set for r in b['relevant_ids']))
print(f'Benchmark: {len(BENCHMARK)} soru | Coverage: {covered}/{len(BENCHMARK)}')

  corpus: OK (22.4 MB)
  gold: OK (0.8 MB)
Corpus yukleniyor...
Corpus: 7643 chunk
Benchmark yukleniyor...
Benchmark: 240 soru | Coverage: 240/240


## 4. FT Model Kontrol

In [5]:
FT_EMBED_PATH = Path(CFG.MODEL_DIR) / 'bgem3'
if not FT_EMBED_PATH.exists():
    raise FileNotFoundError(
        f'FT embedding bulunamadi: {FT_EMBED_PATH}\n'
        'Kaggle > Add Data > /kaggle/input/legal-models-tr-finetuned dataseti eklenmeli!')
print(f'FT BGE-M3 bulundu: {FT_EMBED_PATH}')

FT BGE-M3 bulundu: /kaggle/input/datasets/ardayildiz29/legal-models-tr-finetuned/bgem3


## 5. FAISS (FT BGE-M3)

In [6]:
import faiss, numpy as np
from sentence_transformers import SentenceTransformer

EMBED_PATH = str(FT_EMBED_PATH)
INDEX_PATH = Path(CFG.INDEX_DIR) / 'faiss_s6.bin'
DOCS_PATH  = Path(CFG.INDEX_DIR) / 'docs_s6.pkl'

rebuild = True
if INDEX_PATH.exists() and DOCS_PATH.exists():
    try:
        _idx = faiss.read_index(str(INDEX_PATH))
        with open(DOCS_PATH,'rb') as f: _docs = pickle.load(f)
        if _idx.ntotal == len(CORPUS):
            FAISS_INDEX = _idx; rebuild = False
            print(f'FAISS cache: {FAISS_INDEX.ntotal} vektor (FT BGE-M3)')
        else:
            print('Cache uyusmuyor; yeniden olusturuluyor.')
    except Exception as e:
        print(f'Cache hata: {e}')

if rebuild:
    print(f'Encoder yukleniyor (FT BGE-M3)...')
    _enc  = SentenceTransformer(EMBED_PATH, device=CFG.EMBED_DEVICE)
    texts = [c['text'] for c in CORPUS]
    print(f'{len(texts)} chunk embed ediliyor...')
    embs  = _enc.encode(texts, batch_size=CFG.EMBED_BATCH, normalize_embeddings=True,
                        show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
    dim = embs.shape[1]
    FAISS_INDEX = faiss.IndexFlatIP(dim)
    FAISS_INDEX.add(embs)
    faiss.write_index(FAISS_INDEX, str(INDEX_PATH))
    with open(DOCS_PATH,'wb') as f: pickle.dump(CORPUS, f)
    del _enc, embs; gc.collect()
    print(f'FAISS hazir: {FAISS_INDEX.ntotal} vektor, dim={dim}')

print(f'Query encoder yukleniyor (FT BGE-M3)...')
QUERY_ENCODER = SentenceTransformer(EMBED_PATH, device=CFG.EMBED_DEVICE)
print('Hazir.')

2026-05-31 10:43:15.727118: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780224195.938974      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780224196.001158      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780224196.518702      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780224196.518741      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780224196.518744      58 computation_placer.cc:177] computation placer alr

Encoder yukleniyor (FT BGE-M3)...


The tokenizer you are loading from '/kaggle/input/datasets/ardayildiz29/legal-models-tr-finetuned/bgem3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


7643 chunk embed ediliyor...


Batches:   0%|          | 0/120 [00:00<?, ?it/s]

FAISS hazir: 7643 vektor, dim=1024
Query encoder yukleniyor (FT BGE-M3)...


The tokenizer you are loading from '/kaggle/input/datasets/ardayildiz29/legal-models-tr-finetuned/bgem3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Hazir.


## 6. BM25 Index

In [7]:
from rank_bm25 import BM25Okapi

def simple_tokenize(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return [t for t in text.split() if len(t) > 1]

print('BM25 index olusturuluyor...')
tokenized  = [simple_tokenize(c['text']) for c in CORPUS]
BM25_INDEX = BM25Okapi(tokenized)
print(f'BM25 hazir: {len(tokenized)} belge')

BM25 index olusturuluyor...
BM25 hazir: 7643 belge


## 7. FT Reranker

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
FT_RERANKER_PATH = Path(CFG.MODEL_DIR) / 'reranker_ft'
if not FT_RERANKER_PATH.exists():
    raise FileNotFoundError(f'FT reranker bulunamadi: {FT_RERANKER_PATH}')
_rk_dev      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RK_TOKENIZER = AutoTokenizer.from_pretrained(str(FT_RERANKER_PATH))
RK_MODEL     = AutoModelForSequenceClassification.from_pretrained(
    str(FT_RERANKER_PATH), num_labels=1).to(_rk_dev)
RK_MODEL.eval()
print(f'FT reranker hazir | device={_rk_dev}')

FT reranker hazir | device=cuda


## 8. LLM (Base Qwen)

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print('Base Qwen2.5-7B yukleniyor (4-bit)...')
bnb_inf = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                              bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
max_mem = ({0:'12GiB',1:'13GiB','cpu':'30GiB'}
           if torch.cuda.is_available() and torch.cuda.device_count()>=2 else None)
LLM_MODEL_OBJ = AutoModelForCausalLM.from_pretrained(
    CFG.LLM_MODEL, quantization_config=bnb_inf, device_map='auto',
    max_memory=max_mem, trust_remote_code=True, low_cpu_mem_usage=True, dtype=torch.float16)
LLM_MODEL_OBJ.eval()
LLM_TOKENIZER = AutoTokenizer.from_pretrained(CFG.LLM_MODEL, trust_remote_code=True)
if LLM_TOKENIZER.pad_token is None:
    LLM_TOKENIZER.pad_token = LLM_TOKENIZER.eos_token
LLM_TOKENIZER.padding_side = 'left'
print(f'Base Qwen hazir | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

Base Qwen2.5-7B yukleniyor (4-bit)...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Base Qwen hazir | VRAM: 6.2 GB


## 9. Pipeline

In [10]:
import warnings, numpy as np
warnings.filterwarnings('ignore')

SYSTEM_PROMPT_INF = (
    'Sen bir Turk hukuku RAG asistanisin.\n'
    'Yalnizca kullanicinin verdigi kaynak metne dayanarak cevap ver.\n'
    'Kaynakta olmayan bilgiyi uretme.\n'
    'Cevabinin sonunda kaynak/citation bilgisini mutlaka belirt.'
)

def chunk_citation(c):
    meta = c.get('metadata', {})
    return meta.get('citation_label') or meta.get('chunk_id') or c.get('id','Turk Hukuku')

def build_prompt(query, chunks):
    context = ''
    for i, c in enumerate(chunks, 1):
        meta = c.get('metadata', {}); title = c.get('title') or meta.get('category','Turk Hukuku')
        context += f'[Belge {i} | {title} | Kaynak: {chunk_citation(c)}]\n{c["text"]}\n\n'
    return SYSTEM_PROMPT_INF, f'{context}Soru: {query}\n\nKisa ve kaynakli yanit (2-4 cumle):'

def generate(query, chunks):
    sys_p, user_p = build_prompt(query, chunks)
    msgs = [{'role':'system','content':sys_p},{'role':'user','content':user_p}]
    text = LLM_TOKENIZER.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = LLM_TOKENIZER([text], return_tensors='pt').to(LLM_MODEL_OBJ.device)
    with torch.no_grad():
        out = LLM_MODEL_OBJ.generate(**inputs, max_new_tokens=CFG.MAX_NEW_TOKENS,
            do_sample=False, pad_token_id=LLM_TOKENIZER.eos_token_id, repetition_penalty=1.1)
    return LLM_TOKENIZER.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def dense_retrieve(query, top_k):
    q_emb = QUERY_ENCODER.encode([query], normalize_embeddings=True,
                                  convert_to_numpy=True).astype(np.float32)
    scores, idxs = FAISS_INDEX.search(q_emb, top_k)
    return [dict(**CORPUS[i], dense_score=float(s)) for s, i in zip(scores[0], idxs[0]) if i >= 0]

def sparse_retrieve(query, top_k):
    tokens = simple_tokenize(query)
    scores = BM25_INDEX.get_scores(tokens)
    top_idxs = np.argsort(scores)[::-1][:top_k]
    return [dict(**CORPUS[i], bm25_score=float(scores[i])) for i in top_idxs]

def rrf_fuse(dense, sparse, k=60):
    sc = {}; docs = {}
    for rank, doc in enumerate(dense):
        did = doc['id']; sc[did] = sc.get(did,0.0) + 1.0/(k+rank+1); docs[did] = doc
    for rank, doc in enumerate(sparse):
        did = doc['id']; sc[did] = sc.get(did,0.0) + 1.0/(k+rank+1); docs[did] = doc
    return [dict(**docs[did], rrf_score=round(sc[did],6)) for did in sorted(sc, key=lambda x: sc[x], reverse=True)]

def rerank(query, candidates, top_k):
    if not candidates: return candidates
    enc = RK_TOKENIZER([query]*len(candidates), [c['text'] for c in candidates],
                       truncation=True, padding=True, max_length=CFG.RERANKER_MAX_LEN, return_tensors='pt')
    enc = {k: v.to(next(RK_MODEL.parameters()).device) for k,v in enc.items()}
    with torch.no_grad():
        logits = RK_MODEL(**enc).logits.squeeze(-1)
    ranked = sorted(zip(candidates, logits.cpu().tolist()), key=lambda x: x[1], reverse=True)
    return [dict(**d, reranker_score=round(s,4)) for d,s in ranked[:top_k]]

def rag_pipeline(question, verbose=False):
    dense_res    = dense_retrieve(question, CFG.TOP_K_DENSE)
    sparse_res   = sparse_retrieve(question, CFG.TOP_K_SPARSE)
    fused        = rrf_fuse(dense_res, sparse_res, k=CFG.RRF_K)
    candidates   = fused[:CFG.TOP_K_RERANK_IN]
    all_ids      = [c['id'] for c in candidates]
    final_chunks = rerank(question, candidates, CFG.TOP_K_FINAL)
    answer       = generate(question, final_chunks)
    sources      = list({c.get('title','Turk Hukuku') for c in final_chunks})
    if verbose:
        print(f'Soru: {question}')
        [print(f'  [{i+1}] rk={c.get("reranker_score",0):.4f} rrf={c.get("rrf_score",0):.6f} | {c["text"][:80]}...') for i,c in enumerate(final_chunks)]
        print(f'Cevap: {answer[:300]}')
    return {'question':question,'answer':answer,'retrieved_chunks':final_chunks,'all_retrieved':all_ids,'sources':sources}

print('Pipeline testi (hybrid + reranker)...')
_ = rag_pipeline('Susma hakki nedir?', verbose=True)

Pipeline testi (hybrid + reranker)...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Soru: Susma hakki nedir?
  [1] rk=-0.3469 rrf=0.032018 | Susma hakkı, kişinin kendi lehine veya aleyhine ifade verme kararını özgürce kul...
  [2] rk=-1.5837 rrf=0.015152 | Nemo Tenetur ilkesi, şüphelilere susma hakkını, avukat ile görüşme hakkını ve av...
  [3] rk=-1.7347 rrf=0.031514 | Sanığın susma hakkı, sanığın kendini ifade etme zorunluluğu olmadan sessiz kalma...
  [4] rk=-1.7814 rrf=0.030579 | Yakalanan veya gözaltına alınan bir kişinin susma hakkı, kişinin kendisine yönel...
  [5] rk=-2.1331 rrf=0.030118 | Şüphelinin savunmada hazır bulunma ve savunma hazırlamak için zaman ve imkân tal...
Cevap: Susma hakkı, şüphelinin kendisine yöneltilen sorulardan kimlik bilgisi dışında cevap vermeye zorlanmadan sessiz kalmaya ve savunması için gereken zaman ve imkanları talep etmeye hakkıdır. Bu hakkı, şüphelinin savunmada hazır bulunma ve savunma hazırlamak için zaman ve imkân talep etme hakkıyla birli


## 10. Metrikler

In [11]:
from rouge_score import rouge_scorer as rs_module
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import pandas as pd

def tokenize_tr(text):
    return [t for t in re.sub(r'[^\w\s]',' ',text.lower()).split() if len(t)>1]

def recall_at_k(ret, rel, k):
    if not rel: return None
    return round(len(set(ret[:k])&set(rel))/len(rel), 4)

def mrr_score(ret, rel):
    if not rel: return None
    for rank, did in enumerate(ret, 1):
        if did in set(rel): return round(1.0/rank, 4)
    return 0.0

def ndcg_at_k(ret, rel, k):
    if not rel: return None
    rel_set = set(rel)
    dcg  = sum(1/math.log2(i+2) for i,d in enumerate(ret[:k]) if d in rel_set)
    idcg = sum(1/math.log2(i+2) for i in range(min(len(rel),k)))
    return round(dcg/idcg, 4) if idcg>0 else 0.0

def f1_token(pred, gt):
    p = set(tokenize_tr(pred)); g = set(tokenize_tr(gt))
    if not p or not g: return 0.0
    c = p&g
    if not c: return 0.0
    return round(2*len(c)/(len(p)+len(g)), 4)

def bleu_score(pred, gt):
    return round(sentence_bleu([tokenize_tr(gt)], tokenize_tr(pred),
                 smoothing_function=SmoothingFunction().method1), 4)

def rouge_scores(pred, gt):
    s = rs_module.RougeScorer(['rouge1','rouge2','rougeL']).score(gt, pred)
    return {k: round(s[k].fmeasure, 4) for k in s}

def faithfulness_score(answer, chunks):
    ctx = ' '.join(c['text'] for c in chunks)
    a_t = set(tokenize_tr(answer)); c_t = set(tokenize_tr(ctx))
    return round(len(a_t&c_t)/len(a_t), 4) if a_t else 0.0

def evaluate(benchmark, desc=''):
    rows = []
    for item in tqdm(benchmark, desc=desc or 'Evaluating'):
        result  = rag_pipeline(item['question'])
        ret_ids = [c['id'] for c in result['retrieved_chunks']]
        all_ids = result.get('all_retrieved', ret_ids)
        rel     = item['relevant_ids']; gt = item['answer']
        rg = rouge_scores(result['answer'], gt)
        rows.append({
            'question_id': item.get('question_id'),
            'question'   : item['question'][:80],
            'recall@5'   : recall_at_k(all_ids,rel,5),
            'recall@10'  : recall_at_k(all_ids,rel,10),
            'mrr'        : mrr_score(all_ids,rel),
            'ndcg@5'     : ndcg_at_k(all_ids,rel,5),
            'f1'         : f1_token(result['answer'],gt),
            'bleu'       : bleu_score(result['answer'],gt),
            'rouge1'     : rg['rouge1'], 'rouge2': rg['rouge2'], 'rougeL': rg['rougeL'],
            'faithfulness': faithfulness_score(result['answer'], result['retrieved_chunks']),
            'answer'     : result['answer'][:200],
            'gold_answer': gt[:200],
        })
    df = pd.DataFrame(rows)
    summary = {m: round(df[m].dropna().mean(),4) for m in
               ['recall@5','recall@10','mrr','ndcg@5','f1','bleu','rouge1','rouge2','rougeL','faithfulness']}
    return {'details':df, 'summary':summary}

## 11. Evaluate

In [12]:
print(f'=== {CFG.SYSTEM_NAME} --- {len(BENCHMARK)} soru ===')
EVAL_OUT = evaluate(BENCHMARK, desc=CFG.SYSTEM_NAME)
line = '='*64
print(f'\n{line}\n  {CFG.SYSTEM_NAME}\n  {CFG.SYSTEM_DESC}\n{line}')
print('  RETRIEVAL METRİKLERİ')
for m in ['recall@5','recall@10','mrr','ndcg@5']:
    print(f'    {m:<15s}: {EVAL_OUT["summary"].get(m,"N/A")}')
print('  QA METRİKLERİ')
for m in ['f1','bleu','rouge1','rouge2','rougeL','faithfulness']:
    print(f'    {m:<15s}: {EVAL_OUT["summary"].get(m,"N/A")}')
print(line)
hall = EVAL_OUT['details'][EVAL_OUT['details']['faithfulness'] < 0.3]
print(f'\nHallucination suphelisi (faithfulness<0.3): {len(hall)}/{len(BENCHMARK)}')

=== System6_FTReranker --- 240 soru ===


System6_FTReranker: 100%|██████████| 240/240 [1:23:57<00:00, 20.99s/it]


  System6_FTReranker
  FT BGE-M3 + BM25 + RRF + FT CE + Base Qwen2.5-7B
  RETRIEVAL METRİKLERİ
    recall@5       : 0.8708
    recall@10      : 0.9167
    mrr            : 0.8181
    ndcg@5         : 0.8259
  QA METRİKLERİ
    f1             : 0.3818
    bleu           : 0.1408
    rouge1         : 0.4308
    rouge2         : 0.283
    rougeL         : 0.3391
    faithfulness   : 0.5803

Hallucination suphelisi (faithfulness<0.3): 10/240


## 12. Kaydet

In [13]:
from datetime import datetime
ts       = datetime.now().strftime('%Y%m%d_%H%M%S')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO_GPU'
result_path = f'{CFG.RESULTS_DIR}/{CFG.SYSTEM_NAME}_{ts}.json'
with open(result_path,'w',encoding='utf-8') as f:
    json.dump({'system':CFG.SYSTEM_NAME,'desc':CFG.SYSTEM_DESC,'timestamp':ts,
               'gpu':gpu_name,'benchmark':len(BENCHMARK),'config':asdict(CFG),
               'metrics':EVAL_OUT['summary']}, f, ensure_ascii=False, indent=2)
csv_path = f'{CFG.RESULTS_DIR}/{CFG.SYSTEM_NAME}_details_{ts}.csv'
EVAL_OUT['details'].to_csv(csv_path, index=False, encoding='utf-8')
print(f'Kaydedildi:')
print(f'  Ozet : {result_path}')
print(f'  Detay: {csv_path}')
print(f'\n OK {CFG.SYSTEM_NAME} tamamlandi!')

Kaydedildi:
  Ozet : /kaggle/working/legal_rag/results/System6_FTReranker_20260531_121424.json
  Detay: /kaggle/working/legal_rag/results/System6_FTReranker_details_20260531_121424.csv

 OK System6_FTReranker tamamlandi!


## Interactive

In [14]:
question = "SORUNUZU YAZIN :)"
if question and question != "SORUNUZU YAZIN :)":
    result = rag_pipeline(question, verbose=True)
    print('\n' + '='*60)
    print('CEVAP:', result['answer'])